<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day22_Designing%20Production-Ready%20AI%20APIs/production_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 22 — Designing Production-Ready AI APIs 🛡️

Redesigns the Day 20 endpoint into a production-grade AI API: typed Pydantic models with field validators, input validation with structured error codes, retry logic with exponential backoff, streaming responses, and a 15-second timeout guard.

> **Provider note:** uses **Google Gemini** (`gemini-embedding-001` + `gemini-flash-lite-latest`) instead of OpenAI, due to OpenAI API quota limits hit earlier in the challenge. The retry/backoff logic targets Gemini's `429` rate-limit errors, the same failure mode OpenAI's rate limits would produce.

In [2]:
!pip install -q fastapi "uvicorn[standard]" pydantic numpy faiss-cpu google-generativeai langchain-text-splitters nest_asyncio requests

import os, sys, json, time, uuid, asyncio, threading
from datetime import datetime
from typing import List, Dict, Any, Optional, AsyncGenerator

import numpy as np
import faiss
import google.generativeai as genai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field, field_validator
import uvicorn
import nest_asyncio
import requests

from google.colab import userdata

nest_asyncio.apply()
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
chat_model = genai.GenerativeModel('gemini-flash-lite-latest')
EMBEDDING_MODEL = "models/gemini-embedding-001"

print("Setup done.")

Setup done.


## Rebuild the Day 20 knowledge base + pipeline (same as Day 20/21)

In [3]:
class Document:
    def __init__(self, doc_id, title, text, category, date, document_type):
        self.doc_id = doc_id
        self.title = title
        self.text = text
        self.category = category
        self.date = date
        self.document_type = document_type


class Chunk:
    def __init__(self, chunk_id, text, source_doc):
        self.chunk_id = chunk_id
        self.text = text
        self.source_doc = source_doc

    def to_dict(self):
        return {
            "chunk_id": self.chunk_id, "text": self.text,
            "source_id": self.source_doc.doc_id, "source_title": self.source_doc.title,
        }


DOCUMENTS = [
    Document("doc_company_history", "Company History",
        "Nimbus Robotics was founded in 2031 by engineer Priya Kalathil in Pune, India. "
        "The company employs 212 people across three offices: Pune, Bengaluru, and Singapore.",
        "company_background", "2031-03-15", "reference"),
    Document("doc_aster7_specs", "Aster-7 Product Specifications",
        "Nimbus Robotics' flagship product is the Aster-7, a warehouse picking robot with a "
        "99.2% accuracy rate. The Aster-7 uses a proprietary gripper called FlexGrip. Its "
        "battery lasts 14 hours on a single charge and recharges fully in 40 minutes.",
        "product", "2033-01-10", "technical"),
    Document("doc_finance", "Financial Performance",
        "Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033, an "
        "increase of approximately 65% over the prior year.",
        "finance", "2033-12-31", "reference"),
    Document("doc_competition", "Competitive Landscape",
        "Nimbus Robotics' main competitor is Solace Automation, founded a year earlier in 2030.",
        "market", "2032-06-01", "marketing"),
]


def chunk_documents(documents, chunk_size=500, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in documents:
        pieces = splitter.split_text(doc.text)
        for i, piece in enumerate(pieces):
            chunks.append(Chunk(f"{doc.doc_id}::chunk{i}", piece, doc))
    return chunks


def get_embeddings(texts, max_retries=3):
    embeddings = []
    for text in texts:
        for attempt in range(max_retries):
            try:
                result = genai.embed_content(model=EMBEDDING_MODEL, content=text)
                embeddings.append(result["embedding"])
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(5)
                else:
                    raise RuntimeError(f"Embedding failed: {e}")
        time.sleep(1)
    return embeddings


class RAGIndex:
    def __init__(self, chunks):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        vectors = np.array(get_embeddings(texts), dtype=np.float32)
        self.dimension = vectors.shape[1]
        self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(vectors)

    def search(self, query, top_k=3, search_pool=10):
        query_vector = np.array(get_embeddings([query]), dtype=np.float32)
        distances, indices = self.index.search(query_vector, search_pool)
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx == -1:
                continue
            chunk = self.chunks[idx]
            entry = chunk.to_dict()
            entry["similarity"] = float(1 / (1 + dist))
            results.append(entry)
            if len(results) >= top_k:
                break
        return results


print("Building index...")
_chunks = chunk_documents(DOCUMENTS)
_index = RAGIndex(_chunks)
print(f"Index built with {len(_chunks)} chunks.")

Building index...
Index built with 4 chunks.


## Structured error codes

Three consistent error shapes, each with `code`, `message`, and `request_id`:
- **INPUT_INVALID** — query fails validation (too short, too long, whitespace/symbols only)
- **RETRIEVAL_FAILURE** — the retrieval step itself throws (embedding API failure, index error)
- **LLM_TIMEOUT** — the full pipeline doesn't complete within the timeout window

In [4]:
class ErrorResponse(BaseModel):
    code: str
    message: str
    request_id: str


def make_error(code: str, message: str, request_id: str) -> Dict[str, Any]:
    return ErrorResponse(code=code, message=message, request_id=request_id).model_dump()


ERROR_STATUS_MAP = {
    "INPUT_INVALID": 400,
    "RETRIEVAL_FAILURE": 502,
    "LLM_TIMEOUT": 504,
}

print("Error code definitions ready.")

Error code definitions ready.


## Pydantic request/response models with field validators + schema examples

In [5]:
class AskRequest(BaseModel):
    query: str = Field(
        ...,
        description="The user's question.",
        json_schema_extra={"example": "What is the Aster-7's battery life?"},
    )
    top_k: int = Field(
        default=3, ge=1, le=10,
        description="Number of chunks to retrieve.",
        json_schema_extra={"example": 3},
    )

    @field_validator("query")
    @classmethod
    def validate_query(cls, v: str) -> str:
        stripped = v.strip()

        if len(stripped) < 5:
            raise ValueError("Query must be at least 5 characters long.")
        if len(stripped) > 1000:
            raise ValueError("Query must not exceed 1000 characters.")
        if not any(c.isalnum() for c in stripped):
            raise ValueError("Query must contain at least one letter or number, not just whitespace or symbols.")

        return stripped


class SourceItem(BaseModel):
    source_id: str = Field(..., json_schema_extra={"example": "doc_aster7_specs"})
    title: str = Field(..., json_schema_extra={"example": "Aster-7 Product Specifications"})
    chunk_id: str = Field(..., json_schema_extra={"example": "doc_aster7_specs::chunk0"})
    similarity: float = Field(..., json_schema_extra={"example": 0.71})


class AskResponse(BaseModel):
    answer: str = Field(..., json_schema_extra={"example": "The Aster-7's battery lasts 14 hours [doc_aster7_specs]."})
    sources: List[SourceItem]
    low_confidence: bool
    request_id: str

print("Pydantic models defined.")

Pydantic models defined.


## Retry logic with exponential backoff for rate-limit errors

In [6]:
def call_llm_with_retry(prompt: str, max_retries: int = 4, base_delay: float = 2.0) -> str:
    """Calls the chat model with exponential backoff on rate-limit / transient errors.
    Delay sequence: base_delay * 2^attempt (2s, 4s, 8s, 16s by default)."""
    last_error = None

    for attempt in range(max_retries):
        try:
            response = chat_model.generate_content(
                prompt, generation_config=genai.types.GenerationConfig(temperature=0)
            )
            return response.text.strip()
        except Exception as e:
            last_error = e
            error_str = str(e).lower()
            is_rate_limit = "429" in error_str or "quota" in error_str or "rate" in error_str

            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                print(f"  [retry {attempt+1}/{max_retries}] "
                      f"{'rate limit' if is_rate_limit else 'error'}, backing off {delay:.1f}s: {e}")
                time.sleep(delay)
            else:
                raise RuntimeError(f"LLM call failed after {max_retries} attempts: {last_error}")

    raise RuntimeError(f"LLM call failed: {last_error}")

print("Retry-with-backoff function ready.")

Retry-with-backoff function ready.


## Full pipeline with 15-second timeout guard

In [7]:
import concurrent.futures

PIPELINE_TIMEOUT_SECONDS = 15

GROUNDING_SYSTEM_PROMPT = """You are a strict, grounded knowledge assistant. Answer ONLY
using the context below. If the context doesn't support the answer, say
"I cannot answer this based on the available knowledge base."
Cite sources inline using [source_id] tags.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""

_executor = concurrent.futures.ThreadPoolExecutor(max_workers=4)


def _run_pipeline(query: str, top_k: int, request_id: str) -> Dict[str, Any]:
    """The actual pipeline logic, run in a worker thread so it can be timed out."""
    try:
        retrieved = _index.search(query, top_k=top_k)
    except Exception as e:
        raise RuntimeError(f"RETRIEVAL_FAILURE::{e}")

    if not retrieved:
        return {
            "answer": "I cannot answer this based on the available knowledge base.",
            "sources": [], "low_confidence": True, "request_id": request_id,
        }

    context = "\n".join(f"[{r['source_id']}] {r['text']}" for r in retrieved)
    prompt = GROUNDING_SYSTEM_PROMPT.format(context=context, question=query)

    answer_text = call_llm_with_retry(prompt)

    best_similarity = max(r["similarity"] for r in retrieved)
    low_confidence = best_similarity < 0.3

    sources = [SourceItem(
        source_id=r["source_id"], title=r["source_title"],
        chunk_id=r["chunk_id"], similarity=round(r["similarity"], 4)
    ) for r in retrieved]

    return {"answer": answer_text, "sources": sources,
            "low_confidence": low_confidence, "request_id": request_id}


def run_pipeline_with_timeout(query: str, top_k: int, request_id: str) -> Dict[str, Any]:
    """Runs _run_pipeline with a hard 15-second timeout. Raises TimeoutError or
    a RuntimeError tagged RETRIEVAL_FAILURE on retrieval-specific failures."""
    future = _executor.submit(_run_pipeline, query, top_k, request_id)
    try:
        return future.result(timeout=PIPELINE_TIMEOUT_SECONDS)
    except concurrent.futures.TimeoutError:
        future.cancel()
        raise TimeoutError(f"Pipeline exceeded {PIPELINE_TIMEOUT_SECONDS}s timeout")

print("Timeout-guarded pipeline ready.")

Timeout-guarded pipeline ready.


## FastAPI app: /ask (JSON) and /ask/stream (streaming)

In [8]:
app = FastAPI(
    title="Nimbus Knowledge Assistant — Production API",
    description="Day 22: validated, retried, timed-out, and streamable AI endpoint.",
    version="2.0.0",
)


@app.exception_handler(Exception)
async def generic_exception_handler(request: Request, exc: Exception):
    request_id = str(uuid.uuid4())
    error_str = str(exc)

    if isinstance(exc, TimeoutError):
        body = make_error("LLM_TIMEOUT", "The request took too long to complete.", request_id)
        return JSONResponse(status_code=ERROR_STATUS_MAP["LLM_TIMEOUT"], content=body)

    if "RETRIEVAL_FAILURE" in error_str:
        message = error_str.split("RETRIEVAL_FAILURE::", 1)[-1]
        body = make_error("RETRIEVAL_FAILURE", f"Retrieval step failed: {message}", request_id)
        return JSONResponse(status_code=ERROR_STATUS_MAP["RETRIEVAL_FAILURE"], content=body)

    body = make_error("RETRIEVAL_FAILURE", f"Unexpected error: {error_str}", request_id)
    return JSONResponse(status_code=500, content=body)


@app.get("/")
def health_check():
    return {"status": "ok", "indexed_chunks": len(_chunks)}


@app.post("/ask", response_model=AskResponse, responses={
    400: {"model": ErrorResponse}, 502: {"model": ErrorResponse}, 504: {"model": ErrorResponse}
})
def ask(request: AskRequest):
    request_id = str(uuid.uuid4())
    # Pydantic's field_validator already rejected invalid input before this point.
    # FastAPI turns a ValidationError into a 422 by default; we normalize that to
    # our own INPUT_INVALID shape via the validation exception handler below.
    result = run_pipeline_with_timeout(request.query, request.top_k, request_id)
    return result


from fastapi.exceptions import RequestValidationError

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    request_id = str(uuid.uuid4())
    first_error = exc.errors()[0]
    message = first_error.get("msg", "Invalid input.")
    body = make_error("INPUT_INVALID", message, request_id)
    return JSONResponse(status_code=ERROR_STATUS_MAP["INPUT_INVALID"], content=body)


@app.post("/ask/stream")
async def ask_stream(request: AskRequest):
    """Streams the answer token-by-token-ish (sentence chunks, since Gemini's
    Python SDK streaming API differs from OpenAI's) so the client sees output
    before generation fully completes."""
    request_id = str(uuid.uuid4())

    async def token_stream() -> AsyncGenerator[bytes, None]:
        try:
            retrieved = _index.search(request.query, top_k=request.top_k)
        except Exception as e:
            yield json.dumps(make_error("RETRIEVAL_FAILURE", str(e), request_id)).encode()
            return

        if not retrieved:
            yield b"I cannot answer this based on the available knowledge base."
            return

        context = "\n".join(f"[{r['source_id']}] {r['text']}" for r in retrieved)
        prompt = GROUNDING_SYSTEM_PROMPT.format(context=context, question=request.query)

        try:
            stream = chat_model.generate_content(prompt, stream=True)
            for chunk in stream:
                if chunk.text:
                    yield chunk.text.encode()
                    await asyncio.sleep(0)  # yield control so bytes flush to the client
        except Exception as e:
            yield f"\n[ERROR: {e}]".encode()

    return StreamingResponse(token_stream(), media_type="text/plain")


print("FastAPI app defined with /ask and /ask/stream.")

FastAPI app defined with /ask and /ask/stream.


## Start the server in-notebook

In [9]:
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("Server started at http://127.0.0.1:8000")

Server started at http://127.0.0.1:8000


## Trigger each error path and verify the response

Deliberately sends invalid input for each of the three structured error codes,
printing the response body and HTTP status code to confirm the shape is correct.

In [10]:
print("=" * 60)
print("TEST 1: INPUT_INVALID — query too short (under 5 chars)")
r = requests.post("http://127.0.0.1:8000/ask", json={"query": "hi"})
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

TEST 1: INPUT_INVALID — query too short (under 5 chars)
Status: 400
{
  "code": "INPUT_INVALID",
  "message": "Value error, Query must be at least 5 characters long.",
  "request_id": "34ddb54d-d8ab-4c17-8af3-5c57d50681ff"
}


In [11]:
print("=" * 60)
print("TEST 2: INPUT_INVALID — query too long (over 1000 chars)")
r = requests.post("http://127.0.0.1:8000/ask", json={"query": "a" * 1001})
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

TEST 2: INPUT_INVALID — query too long (over 1000 chars)
Status: 400
{
  "code": "INPUT_INVALID",
  "message": "Value error, Query must not exceed 1000 characters.",
  "request_id": "2148d390-e7b5-4dbe-87fb-33ae66586411"
}


In [12]:
print("=" * 60)
print("TEST 3: INPUT_INVALID — whitespace/symbols only")
r = requests.post("http://127.0.0.1:8000/ask", json={"query": "!!!   ???"})
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

TEST 3: INPUT_INVALID — whitespace/symbols only
Status: 400
{
  "code": "INPUT_INVALID",
  "message": "Value error, Query must contain at least one letter or number, not just whitespace or symbols.",
  "request_id": "add6aa43-a023-42d1-983d-45b2ef192220"
}


In [13]:
print("=" * 60)
print("TEST 4: LLM_TIMEOUT — force a timeout by setting PIPELINE_TIMEOUT_SECONDS very low")

# Temporarily patch the timeout to 0.01s so the pipeline is guaranteed to exceed it,
# without needing to actually wait ~15 real seconds to prove the timeout path works.
import builtins
original_timeout = PIPELINE_TIMEOUT_SECONDS

def run_pipeline_forced_timeout(query, top_k, request_id):
    future = _executor.submit(_run_pipeline, query, top_k, request_id)
    try:
        return future.result(timeout=0.01)  # force timeout
    except concurrent.futures.TimeoutError:
        future.cancel()
        raise TimeoutError(f"Pipeline exceeded 0.01s forced timeout")

# Monkey-patch for this test only
import sys
_orig = run_pipeline_with_timeout
globals()['run_pipeline_with_timeout'] = run_pipeline_forced_timeout

r = requests.post("http://127.0.0.1:8000/ask", json={"query": "What is the Aster-7's battery life?"})
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

# Restore real timeout for subsequent tests
globals()['run_pipeline_with_timeout'] = _orig
print(f"\nTimeout restored to {PIPELINE_TIMEOUT_SECONDS}s for normal operation.")

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/tmp/ipykernel_549/3136364803.py", line 12, in run_pipeline_forced_timeout
    return future.result(timeout=0.01)  # force timeout
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/concurrent/futures/_base.py", line 462, in result
    raise TimeoutError()
TimeoutError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^

TEST 4: LLM_TIMEOUT — force a timeout by setting PIPELINE_TIMEOUT_SECONDS very low
Status: 504
{
  "code": "LLM_TIMEOUT",
  "message": "The request took too long to complete.",
  "request_id": "4e4dabcb-1fb0-4bd6-8f02-959197e10db4"
}

Timeout restored to 15s for normal operation.


In [14]:
print("=" * 60)
print("TEST 5: RETRIEVAL_FAILURE — force the index to throw")

_original_search = _index.search
def broken_search(*args, **kwargs):
    raise RuntimeError("Simulated FAISS index corruption")
_index.search = broken_search

r = requests.post("http://127.0.0.1:8000/ask", json={"query": "What is the Aster-7's battery life?"})
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

# Restore real search
_index.search = _original_search
print("\nRetrieval restored to normal operation.")

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/tmp/ipykernel_549/510609173.py", line 24, in _run_pipeline
    retrieved = _index.search(query, top_k=top_k)
  File "/tmp/ipykernel_549/590346912.py", line 6, in broken_search
    raise RuntimeError("Simulated FAISS index corruption")
RuntimeError: Simulated FAISS index corruption

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

TEST 5: RETRIEVAL_FAILURE — force the index to throw
Status: 502
{
  "code": "RETRIEVAL_FAILURE",
  "message": "Retrieval step failed: Simulated FAISS index corruption",
  "request_id": "01ccf1a6-de6e-42fb-8fb3-3fa7f2ee1e3d"
}

Retrieval restored to normal operation.


In [16]:
print("=" * 60)
print("TEST 6: valid request — confirming normal operation still works after error tests")
r = requests.post("http://127.0.0.1:8000/ask", json={"query": "What is the Aster-7's battery life?"})
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

TEST 6: valid request — confirming normal operation still works after error tests


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/tmp/ipykernel_549/510609173.py", line 56, in run_pipeline_with_timeout
    return future.result(timeout=PIPELINE_TIMEOUT_SECONDS)
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/concurrent/futures/_base.py", line 462, in result
    raise TimeoutError()
TimeoutError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^

Status: 504
{
  "code": "LLM_TIMEOUT",
  "message": "The request took too long to complete.",
  "request_id": "7d795b1f-4fba-45ab-9f42-00737a8429b3"
}


## Test the streaming endpoint

In [17]:
print("Streaming response (tokens arrive incrementally):\n")

with requests.post(
    "http://127.0.0.1:8000/ask/stream",
    json={"query": "What is the Aster-7's battery life and gripper called?"},
    stream=True
) as r:
    for chunk in r.iter_content(chunk_size=None, decode_unicode=True):
        if chunk:
            print(chunk, end="", flush=True)
print("\n\nStream complete.")

Streaming response (tokens arrive incrementally):

{"code": "RETRIEVAL_FAILURE", "message": "Embedding failed: HTTPConnectionPool(host='localhost', port=44057): Read timed out. (read timeout=60.0)", "request_id": "eb516724-4d2f-46dc-a33e-8685f924e12a"}

Stream complete.
